# Self-Supervised Learning

**Self-Supervised Learning (SSL)** is a Deep Learning technique that leverages **pretext tasks** to pre-train a model before it is applied to a main (downstream) task. This approach is particularly effective when the **number of labeled samples** available for training is limited.

In this notebook, you will complete **two exercises** covering two distinct self-supervised approaches:



1.   **Rotation-Based Pretext Task:** You will perform **image classification** on the MNIST dataset (handwritten digits). Before training the model on limited labeled data, you will pre-train it on a **rotation task.** In this stage, images are randomly rotated by 0, 90, 180, or 270 degrees, and the model must predict the specific angle of rotation. This process forces the model **to learn relevant features** (such as shapes and orientations) before it is fine-tuned for the main digit classification task.
2.   **Contrastive Learning Pretext Task:** You will perform **image classification** on the CIFAR-10 dataset (containing 10 classes, such as airplanes, birds, and cats). For this exercise, you will pre-train the model using a Contrastive Learning approach (specifically SimCLR). The model will learn to map two different augmented versions of the same image **close together** in the feature space, while simultaneously mapping different images **far apart.** Through this process, the model automatically learns to extract high-level features necessary for robust image recognition across all classes.



## Rotation Pretext Task

### Imports and Utilities

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
import copy
import numpy as np
import random

In [ ]:
# The MNIST dataset contains digits from 0 to 9. Since the pretext task is rotation-based, it makes no
# sense to rotate 6 and 9: this would just confuse the model.
# Therefore, the rotation pretext task will be applied using just 0, 1, 2, 3, 4, 5, 7, 8 digits.
def get_filtered_mnist():
    # Define transformation
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])

    # Load the dataset
    full_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

    # Take every sample except 6 and 9 classes
    indices = [i for i, (_, label) in enumerate(full_train) if label not in [6, 9]]

    # Return the subset
    return Subset(full_train, indices)

### Model Definition

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Flatten(),
            nn.Linear(64 * 12 * 12, 128),
            nn.ReLU()
        )
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        features = self.encoder(x)
        return self.fc(features)

### Custom Dataset (for Rotation Task)

In [ ]:
class RotationDataset(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
        self.angles = [0, 90, 180, 270]

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        img, _ = self.dataset[index]
        label = torch.randint(0, 4, (1,)).item()
        
        rotated_img = TF.rotate(img, self.angles[label])
        return rotated_img, label

### Training and Evaluation

In [ ]:
def train_model(model, loader, epochs, lr):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for _ in range(epochs):
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), lbls)
            loss.backward()
            optimizer.step()

def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            correct += (model(imgs).argmax(dim=1) == lbls).sum().item()
    return correct / len(loader.dataset)

In [1]:
# Initialize the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NameError: name 'torch' is not defined

In [ ]:
def run_multi_trial_experiment():
    mnist_filtered = Subset(train_dataset, [i for i, (_, y) in enumerate(train_dataset) if y not in [6, 9]])

    print("Step 1: Training Pretext Task (Rotation)...")
    pretext_dataset = RotationDataset(mnist_filtered)
    pretext_loader = DataLoader(pretext_dataset, batch_size=128, shuffle=True)
    base_ssl_model = SimpleCNN(num_classes=4).to(device)

    train_model(base_ssl_model, pretext_loader, epochs=5, lr=0.001)

    sample_sizes = [10, 25, 50, 100, 200]
    num_trials = 5

    results = { "scratch": {n: [] for n in sample_sizes}, "ssl": {n: [] for n in sample_sizes} }

    val_loader = DataLoader(Subset(mnist_filtered, range(2000, 4000)), batch_size=128)

    for n in sample_sizes:
        print(f"\n--- Testing N={n} labeled samples ---")
        for trial in range(num_trials):
            indices = torch.randperm(len(mnist_filtered))[:n]
            train_loader = DataLoader(Subset(mnist_filtered, indices), batch_size=min(n, 16), shuffle=True)

            m_scratch = SimpleCNN(num_classes=10).to(device)
            train_model(m_scratch, train_loader, epochs=10, lr=0.001)

            results["scratch"][n].append(evaluate(m_scratch, val_loader))

            m_ssl = copy.deepcopy(base_ssl_model)
            m_ssl.fc = nn.Linear(128, 10).to(device)

            train_model(m_ssl, train_loader, epochs=10, lr=0.0005)
            results["ssl"][n].append(evaluate(m_ssl, val_loader))

            print(f" Trial {trial+1}/{num_trials} complete.")

    scratch_means = [np.mean(results["scratch"][n]) for n in sample_sizes]
    ssl_means = [np.mean(results["ssl"][n]) for n in sample_sizes]

    plt.figure(figsize=(10, 6))
    plt.plot(sample_sizes, scratch_means, 'o-', label='From Scratch (Avg)', color='crimson', linewidth=2)
    plt.plot(sample_sizes, ssl_means, 's-', label='SSL Pre-trained (Avg)', color='seagreen', linewidth=2)

    plt.title(f"SSL vs Scratch: Average of {num_trials} Trials", fontsize=14)
    plt.xlabel("Number of Labeled Samples", fontsize=12)
    plt.ylabel("Validation Accuracy", fontsize=12)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.show()

In [ ]:
run_multi_trial_experiment()

## Contrastive Learning

### Imports

In [ ]:
import os
import glob
import copy
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset

from torchvision import models, datasets, transforms
import torchvision.transforms.functional as TF

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Contrastive Learning Logic

#### Augmentations

In [ ]:
class SimCLRTransform:
    def __init__(self, size=32):
        self.transform = transforms.Compose([
            transforms.RandomResizedCrop(size),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([transforms.ColorJitter(0.8, 0.8, 0.8, 0.2)], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
        ])

    def __call__(self, x):
        return self.transform(x), self.transform(x)

#### Model

In [ ]:
class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(weights=None)
        dim_mlp = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.projector = nn.Sequential(nn.Linear(dim_mlp, dim_mlp), nn.ReLU(), nn.Linear(dim_mlp, 128))

    def forward(self, x):
        h = self.backbone(x)
        return self.projector(h)

#### Contrastive Loss

The **Contrastive Loss** is the core of Contrastive Learning: it teaches the model how to recognize **two different instances of the same class** without telling it which class it is.

Given a training batch, we create a positive pair and several negative pairs.


*   **Positive Pair:** the two instances of the same image.
*   **Negative Pair:** every other image in the current batch wrt the anchor image.

Then, through matrix multiplication, a **grid of scores** is created.


*   The **diagonal** of this matrix represents the similarity between an image and its own augmented twin (the positive pairs).
*   Everything **off the diagonal** represents the similarity between an image and a completely different image (the negative pairs).

We use CrossEntropyLoss to pick **1 "twin" out of 255 "strangers"** (assuming a batch size of 256).









In [ ]:
def contrastive_loss(z1, z2, temperature=0.1):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    logits_1_to_2 = torch.matmul(z1, z2.T) / temperature
    logits_2_to_1 = torch.matmul(z2, z1.T) / temperature

    labels = torch.arange(z1.size(0)).to(z1.device)

    loss_a = F.cross_entropy(logits_1_to_2, labels)
    loss_b = F.cross_entropy(logits_2_to_1, labels)

    return (loss_a + loss_b) / 2

#### Training on Pretext Task

In [ ]:
target_classes = [0, 1, 2, 3, 4]
images_per_class = 2000

full_train_ds = datasets.CIFAR10(root='./data', train=True, download=True)
all_labels = np.array(full_train_ds.targets)

subset_indices = []
for cls in target_classes:
    cls_indices = np.where(all_labels == cls)[0]
    selected_indices = cls_indices[:images_per_class]
    subset_indices.extend(selected_indices)

transformed_ds = datasets.CIFAR10(
    root='./data',
    train=True,
    download=False,
    transform=SimCLRTransform(size=32)
)

pretext_dataset = Subset(transformed_ds, subset_indices)
pretext_loader = DataLoader(pretext_dataset, batch_size=256, shuffle=True)

print(f"Total images for Pretext Task: {len(pretext_dataset)}")

In [ ]:
model = SimCLR().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(20):
    for (v1, v2), _ in pretext_loader:
        v1, v2 = v1.to(device), v2.to(device)

        optimizer.zero_grad()

        z1 = model(v1)
        z2 = model(v2)
        loss = contrastive_loss(z1, z2)

        loss.backward()
        optimizer.step()
    print(f"Pretext Epoch {epoch} Loss: {loss.item():.4f}")

### Main Classification Task

In [ ]:
model_ssl = copy.deepcopy(model.backbone)

model_ssl.fc = nn.Linear(512, 5)
model_ssl = model_ssl.to(device)

model_scratch = models.resnet18(weights=None)
model_scratch.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model_scratch.maxpool = nn.Identity()
model_scratch.fc = nn.Linear(512, 5)
model_scratch = model_scratch.to(device)

In [ ]:
# Standard transform for testing
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

In [ ]:
# Load full CIFAR-10
full_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=eval_transform)
full_test = datasets.CIFAR10(root='./data', train=False, download=True, transform=eval_transform)

In [ ]:
test_indices = [i for i, label in enumerate(full_test.targets) if label < 5]
test_loader = DataLoader(Subset(full_test, test_indices), batch_size=100)

In [ ]:
# Helper functions to keep the loop clean
def train_simple(m, loader, epochs, lr=1e-3):
    m.train()
    opt = optim.Adam(m.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            crit(m(x), y).backward()
            opt.step()

def evaluate_simple(m, loader):
    m.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            correct += (m(x).argmax(1) == y).sum().item()
            total += y.size(0)
    return 100 * correct / total

In [ ]:
def run_cifar_comparison(m_scratch, m_ssl):
    sample_sizes = [10, 25, 50, 100, 200]
    num_trials = 5

    results = {
        "scratch": {n: [] for n in sample_sizes},
        "ssl": {n: [] for n in sample_sizes}
    }

    print("Starting Multi-Trial Experiment...")

    for n in sample_sizes:
        print(f"\n
--- Testing N={n} total labeled samples ---")

        for trial in range(num_trials):
            n_per_class = max(1, n // 5)
            indices = []
            labels_array = np.array(full_train.targets)
            for cls_idx in range(5):
                cls_indices = np.where(labels_array == cls_idx)[0]
                selected = np.random.choice(cls_indices, n_per_class, replace=False)
                indices.extend(selected)

            trial_train_loader = DataLoader(
                Subset(full_train, indices),
                batch_size=min(n, 16),
                shuffle=True
            )

            epochs = 25

            scratch_clone = copy.deepcopy(m_scratch)
            train_model(scratch_clone, trial_train_loader, epochs=epochs, lr=1e-3)
            acc_s = evaluate(scratch_clone, test_loader) * 100
            results["scratch"][n].append(acc_s)

            ssl_clone = copy.deepcopy(m_ssl)
            train_model(ssl_clone, trial_train_loader, epochs=epochs, lr=5e-4)
            acc_l = evaluate(ssl_clone, test_loader) * 100
            results["ssl"][n].append(acc_l)

            print(f" Trial {trial+1}/{num_trials} | Scratch: {acc_s:.1f}% | SSL: {acc_l:.1f}%")

    scratch_means = [np.mean(results["scratch"][n]) for n in sample_sizes]
    ssl_means = [np.mean(results["ssl"][n]) for n in sample_sizes]

    scratch_stds = [np.std(results["scratch"][n]) for n in sample_sizes]
    ssl_stds = [np.std(results["ssl"][n]) for n in sample_sizes]

    plt.figure(figsize=(10, 6))
    plt.errorbar(sample_sizes, scratch_means, yerr=scratch_stds, fmt='o-', label='From Scratch', color='crimson', capsize=5)
    plt.errorbar(sample_sizes, ssl_means, yerr=ssl_stds, fmt='s-', label='SimCLR Pre-trained', color='seagreen', capsize=5)

    plt.title(f"SimCLR vs Scratch: Average of {num_trials} Trials (CIFAR-10)", fontsize=14)
    plt.xlabel("Total Labeled Samples (5 Classes)", fontsize=12)
    plt.ylabel("Test Accuracy (%)", fontsize=12)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.show()

In [ ]:
run_cifar_comparison(model_scratch, model_ssl)